# 26: Building a Transformer from Scratch

## The Architecture That Changed Everything

The Transformer architecture (2017) revolutionized NLP and beyond. Let's build one from scratch to understand every component!

### The Web Dev Analogy

Think of a Transformer like a **modern web application**:
- **Attention** = Database queries ("find relevant info")
- **Multi-head attention** = Multiple indexes (different ways to look up data)
- **Feed-forward layers** = Business logic processors
- **Layer normalization** = Input sanitization
- **Positional encoding** = Adding timestamps to stateless requests

```
Input Tokens --> [Embedding + Position] --> [Attention --> FFN] x N --> Output
```

## What You'll Learn
- [ ] Implement multi-head attention from scratch in PyTorch
- [ ] Build a complete transformer encoder block
- [ ] Train the transformer on a simple sequence task

## How This Differs from Lesson 25

| Lesson 25 (Architecture) | Lesson 26 (From Scratch) |
|--------------------------|-------------------------|
| Used pre-built components | Every line written by you |
| Focus: understand the blueprint | Focus: build the machine |
| "What does each part do?" | "How does each part work internally?" |
| Trained a tiny classifier | Train a sequence-to-sequence model |

**Think of it this way:** Lesson 25 was reading the IKEA manual. This lesson is building the furniture with your own hands.

If you completed Lesson 25, you already understand the architecture. Now you'll prove it by building every piece from scratch.

## Connection to Previous Lessons

| What you learned | How it connects here |
|-----------------|---------------------|
| **Lesson 25**: Transformer architecture (theory) | Now we implement every component in code — attention, feed-forward, residuals |
| **Lesson 12**: PyTorch training loop | Same training pattern: forward → loss → backward → step |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import math

plt.style.use('seaborn-v0_8-whitegrid')
torch.manual_seed(42)

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print("Ready to build a Transformer!")

## 1. Positional Encoding

Transformers process all tokens in parallel, so they have no inherent sense of position. We need to inject position information!

**Key insight**: Use sinusoidal functions at different frequencies. Each position gets a unique "fingerprint".

$$PE_{(pos, 2i)} = \sin(pos / 10000^{2i/d_{model}})$$
$$PE_{(pos, 2i+1)} = \cos(pos / 10000^{2i/d_{model}})$$

In [ ]:
class PositionalEncoding(nn.Module):
    """Inject position information using sinusoidal functions."""
    
    def __init__(self, d_model, max_seq_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        # Create positional encoding matrix
        pe = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        
        # Compute the div term: 10000^(2i/d_model)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        # Apply sin to even indices, cos to odd indices
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # Add batch dimension and register as buffer (not a parameter)
        pe = pe.unsqueeze(0)  # (1, max_seq_len, d_model)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        # x: (batch_size, seq_len, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

# Test it
d_model = 64
pe = PositionalEncoding(d_model, max_seq_len=100)
print(f"Positional encoding shape: {pe.pe.shape}")

In [ ]:
# Visualize positional encodings
plt.figure(figsize=(14, 6))

# Plot 1: Heatmap of positional encodings
plt.subplot(1, 2, 1)
pe_values = pe.pe.squeeze(0).numpy()[:50, :32]  # First 50 positions, 32 dimensions
plt.imshow(pe_values, cmap='RdBu', aspect='auto')
plt.colorbar(label='Value')
plt.xlabel('Embedding Dimension')
plt.ylabel('Position')
plt.title('Positional Encoding Heatmap')

# Plot 2: Show specific dimensions across positions
plt.subplot(1, 2, 2)
positions = np.arange(100)
for dim in [0, 4, 8, 16]:
    plt.plot(positions, pe.pe.squeeze(0)[:, dim].numpy(), label=f'Dim {dim}')
plt.xlabel('Position')
plt.ylabel('Encoding Value')
plt.title('Positional Encoding by Dimension')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Each position has a unique 'fingerprint' of sinusoids!")
print("Lower dimensions = higher frequency, Higher dimensions = lower frequency")

## 2. Scaled Dot-Product Attention

The core of the Transformer! For each token, attention answers: "What other tokens should I pay attention to?"

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

**Web Dev Analogy**: Like a database query:
- **Query (Q)**: What you're searching for
- **Key (K)**: Index to match against
- **Value (V)**: The actual data to retrieve

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Compute attention scores and apply to values.
    
    Args:
        Q: Queries (batch, heads, seq_len, d_k)
        K: Keys (batch, heads, seq_len, d_k)
        V: Values (batch, heads, seq_len, d_v)
        mask: Optional mask (batch, 1, 1, seq_len) or (batch, 1, seq_len, seq_len)
    
    Returns:
        output: Weighted values (batch, heads, seq_len, d_v)
        attention_weights: Attention scores (batch, heads, seq_len, seq_len)
    """
    d_k = Q.size(-1)
    
    # Step 1: Compute attention scores
    # Q @ K^T: (batch, heads, seq_len, d_k) @ (batch, heads, d_k, seq_len)
    #        = (batch, heads, seq_len, seq_len)
    scores = torch.matmul(Q, K.transpose(-2, -1))
    
    # Step 2: Scale by sqrt(d_k) to prevent softmax saturation
    scores = scores / math.sqrt(d_k)
    
    # Step 3: Apply mask (for padding or causal attention)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    # Step 4: Softmax to get attention weights
    attention_weights = F.softmax(scores, dim=-1)
    
    # Step 5: Apply attention to values
    output = torch.matmul(attention_weights, V)
    
    return output, attention_weights

# Demonstrate with a simple example
batch_size, num_heads, seq_len, d_k = 1, 1, 4, 8
Q = torch.randn(batch_size, num_heads, seq_len, d_k)
K = torch.randn(batch_size, num_heads, seq_len, d_k)
V = torch.randn(batch_size, num_heads, seq_len, d_k)

output, weights = scaled_dot_product_attention(Q, K, V)
print(f"Q, K, V shape: {Q.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {weights.shape}")
print(f"\nAttention weights (each row sums to 1):")
print(weights.squeeze().numpy().round(3))

In [ ]:
# Visualize attention: "The cat sat on the mat"
words = ['The', 'cat', 'sat', 'on']

# Simulate meaningful attention (manually set for visualization)
# "sat" should attend to "cat" (subject), "on" attends to "sat" (verb)
manual_weights = torch.tensor([[
    [[0.7, 0.1, 0.1, 0.1],   # The -> mostly self
     [0.2, 0.6, 0.1, 0.1],   # cat -> self, some The
     [0.1, 0.5, 0.3, 0.1],   # sat -> cat (who sat?)
     [0.1, 0.2, 0.4, 0.3]]   # on -> sat (on what action?)
]])

plt.figure(figsize=(8, 6))
plt.imshow(manual_weights.squeeze().numpy(), cmap='Blues')
plt.colorbar(label='Attention Weight')
plt.xticks(range(4), words)
plt.yticks(range(4), words)
plt.xlabel('Key (attending to)')
plt.ylabel('Query (from)')
plt.title('Attention Pattern: "The cat sat on"')

# Add text annotations
for i in range(4):
    for j in range(4):
        plt.text(j, i, f'{manual_weights[0,0,i,j]:.1f}', 
                ha='center', va='center', fontsize=12)

plt.tight_layout()
plt.show()

print("Each row shows how much each word attends to other words.")
print("'sat' strongly attends to 'cat' - finding its subject!")

## 3. Multi-Head Attention

Instead of one attention, use **multiple attention heads** in parallel. Each head can learn different relationships!

**Analogy**: Like having multiple database indexes:
- Head 1: Looks for subjects
- Head 2: Looks for objects
- Head 3: Looks for modifiers
- etc.

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-head attention mechanism."""
    
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # Dimension per head
        
        # Linear projections for Q, K, V
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        
        # Output projection
        self.W_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        
        # 1. Linear projections
        Q = self.W_q(query)  # (batch, seq_len, d_model)
        K = self.W_k(key)
        V = self.W_v(value)
        
        # 2. Reshape to (batch, num_heads, seq_len, d_k)
        Q = Q.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        # 3. Scaled dot-product attention
        attn_output, attn_weights = scaled_dot_product_attention(Q, K, V, mask)
        
        # 4. Concatenate heads: (batch, seq_len, d_model)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        
        # 5. Final linear projection
        output = self.W_o(attn_output)
        
        return output, attn_weights

# Test multi-head attention
d_model = 64
num_heads = 8
seq_len = 10
batch_size = 2

mha = MultiHeadAttention(d_model, num_heads)
x = torch.randn(batch_size, seq_len, d_model)
output, weights = mha(x, x, x)  # Self-attention

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {weights.shape}")
print(f"  ({num_heads} heads, each looking at {seq_len}x{seq_len} attention pattern)")

In [ ]:
# Visualize different attention heads
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
weights_np = weights[0].detach().numpy()  # First sample

for head_idx, ax in enumerate(axes.flatten()):
    im = ax.imshow(weights_np[head_idx], cmap='Blues', vmin=0, vmax=1)
    ax.set_title(f'Head {head_idx + 1}')
    ax.set_xlabel('Key Position')
    ax.set_ylabel('Query Position')

plt.suptitle('Multi-Head Attention: Each Head Learns Different Patterns', fontsize=14)
plt.tight_layout()
plt.show()

print("Each head can specialize in different types of relationships!")

## 4. Feed-Forward Network

After attention, each position passes through a feed-forward network **independently**.

$$\text{FFN}(x) = \text{ReLU}(xW_1 + b_1)W_2 + b_2$$

**Analogy**: Each position runs through the same "business logic" processor.

In [ ]:
class FeedForward(nn.Module):
    """Position-wise feed-forward network."""
    
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        self.activation = nn.GELU()  # Modern transformers use GELU
        
    def forward(self, x):
        # x: (batch, seq_len, d_model)
        x = self.linear1(x)      # (batch, seq_len, d_ff)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.linear2(x)      # (batch, seq_len, d_model)
        return x

# Test it
d_model = 64
d_ff = 256  # Usually 4x d_model

ff = FeedForward(d_model, d_ff)
x = torch.randn(2, 10, d_model)
output = ff(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"\nFFN expands to {d_ff} dims internally, then projects back to {d_model}")
print(f"Parameters: {sum(p.numel() for p in ff.parameters()):,}")

## 5. Layer Normalization

Normalize across features (not batch) for stable training.

**Analogy**: Like sanitizing inputs - keeps values in a reasonable range.

In [ ]:
# PyTorch provides LayerNorm, let's understand it
x = torch.tensor([[1.0, 2.0, 3.0, 4.0, 5.0]])

# Manual layer norm
mean = x.mean(dim=-1, keepdim=True)
std = x.std(dim=-1, keepdim=True, unbiased=False)
manual_norm = (x - mean) / (std + 1e-6)

# PyTorch layer norm
ln = nn.LayerNorm(5, elementwise_affine=False)
pytorch_norm = ln(x)

print(f"Original: {x.numpy()}")
print(f"Manual LayerNorm: {manual_norm.numpy().round(3)}")
print(f"PyTorch LayerNorm: {pytorch_norm.detach().numpy().round(3)}")
print(f"\nMean: {manual_norm.mean().item():.6f} (should be ~0)")
print(f"Std: {manual_norm.std().item():.6f} (should be ~1)")

## 6. Transformer Encoder Block

Now let's combine everything into a single encoder block!

```
Input --> LayerNorm --> Multi-Head Attention --> + (residual)
  |                                              |
  +--------------------------------------------->+
                                                 |
                                                 v
                              LayerNorm --> Feed-Forward --> + (residual)
                                |                           |
                                +-------------------------->+
                                                            |
                                                            v
                                                         Output
```

In [ ]:
class TransformerEncoderBlock(nn.Module):
    """A single Transformer encoder block."""
    
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        
        # Multi-head attention
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        
        # Feed-forward
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        # Self-attention with residual connection (Pre-LN variant)
        normalized = self.norm1(x)
        attn_output, attn_weights = self.attention(normalized, normalized, normalized, mask)
        x = x + self.dropout1(attn_output)
        
        # Feed-forward with residual connection
        normalized = self.norm2(x)
        ff_output = self.ff(normalized)
        x = x + self.dropout2(ff_output)
        
        return x, attn_weights

# Test it
encoder_block = TransformerEncoderBlock(d_model=64, num_heads=8, d_ff=256)
x = torch.randn(2, 10, 64)
output, weights = encoder_block(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"\nComponents: Attention -> FFN with LayerNorm and Residuals")

## 7. Transformer Decoder Block

The decoder has **two** attention mechanisms:
1. **Masked self-attention**: Can only attend to previous positions (for autoregressive generation)
2. **Cross-attention**: Attends to encoder output

In [ ]:
def create_causal_mask(seq_len):
    """Create a causal (look-ahead) mask for autoregressive decoding."""
    # Lower triangular matrix: position i can only attend to positions <= i
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask.unsqueeze(0).unsqueeze(0)  # (1, 1, seq_len, seq_len)

# Visualize causal mask
causal_mask = create_causal_mask(6)
plt.figure(figsize=(6, 5))
plt.imshow(causal_mask.squeeze().numpy(), cmap='Greens')
plt.colorbar(label='Can Attend')
plt.xlabel('Key Position (attending to)')
plt.ylabel('Query Position (from)')
plt.title('Causal Mask: Each Position Only Sees Previous')

for i in range(6):
    for j in range(6):
        val = int(causal_mask[0, 0, i, j].item())
        color = 'white' if val == 1 else 'black'
        plt.text(j, i, str(val), ha='center', va='center', fontsize=14, color=color)

plt.tight_layout()
plt.show()

print("Position 0 can only see itself")
print("Position 3 can see positions 0, 1, 2, 3")
print("This prevents 'cheating' during generation!")

In [ ]:
class TransformerDecoderBlock(nn.Module):
    """A single Transformer decoder block."""
    
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        
        # Masked self-attention
        self.self_attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        
        # Cross-attention (attends to encoder output)
        self.cross_attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout(dropout)
        
        # Feed-forward
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout3 = nn.Dropout(dropout)
        
    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        # 1. Masked self-attention
        normalized = self.norm1(x)
        self_attn_output, _ = self.self_attention(normalized, normalized, normalized, tgt_mask)
        x = x + self.dropout1(self_attn_output)
        
        # 2. Cross-attention to encoder
        normalized = self.norm2(x)
        cross_attn_output, cross_attn_weights = self.cross_attention(
            normalized, encoder_output, encoder_output, src_mask
        )
        x = x + self.dropout2(cross_attn_output)
        
        # 3. Feed-forward
        normalized = self.norm3(x)
        ff_output = self.ff(normalized)
        x = x + self.dropout3(ff_output)
        
        return x, cross_attn_weights

# Test decoder block
decoder_block = TransformerDecoderBlock(d_model=64, num_heads=8, d_ff=256)
encoder_output = torch.randn(2, 10, 64)  # From encoder
decoder_input = torch.randn(2, 5, 64)    # Target sequence (shorter)
causal = create_causal_mask(5)

output, cross_weights = decoder_block(decoder_input, encoder_output, tgt_mask=causal)
print(f"Encoder output: {encoder_output.shape}")
print(f"Decoder input: {decoder_input.shape}")
print(f"Decoder output: {output.shape}")
print(f"Cross-attention: decoder attends to encoder!")

## 8. Complete Transformer Model

Now let's put it all together into a full encoder-decoder Transformer!

In [ ]:
# === Building the Transformer: Part 1/3 — Architecture Setup ===
# Define the Transformer class with all its components:
# embeddings, encoder stack, decoder stack, output projection

class Transformer(nn.Module):
    """Complete Transformer for sequence-to-sequence tasks."""
    
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        d_model=256,
        num_heads=8,
        num_encoder_layers=3,
        num_decoder_layers=3,
        d_ff=1024,
        max_seq_len=512,
        dropout=0.1
    ):
        super().__init__()
        
        # Embeddings: convert token IDs to vectors
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_len, dropout)
        
        # Encoder: stack of self-attention + FFN blocks
        self.encoder_layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_encoder_layers)
        ])
        self.encoder_norm = nn.LayerNorm(d_model)
        
        # Decoder: stack of masked-self-attn + cross-attn + FFN blocks
        self.decoder_layers = nn.ModuleList([
            TransformerDecoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_decoder_layers)
        ])
        self.decoder_norm = nn.LayerNorm(d_model)
        
        # Output projection: decoder hidden states -> vocab logits
        self.output_projection = nn.Linear(d_model, tgt_vocab_size)
        
        # Scale factor for embeddings (from the paper)
        self.d_model = d_model
        self.scale = math.sqrt(d_model)

print("✓ Transformer __init__ defined (Part 1/3)")
print("  Components: embeddings, encoder stack, decoder stack, output projection")

In [ ]:
# === Building the Transformer: Part 2/3 — Encode & Decode Methods ===
# encode(): processes source sequence through the encoder stack
# decode(): processes target sequence through the decoder stack (with cross-attention)

def _encode(self, src, src_mask=None):
    """Encode source sequence into rich representations."""
    # Embed and add positional encoding
    x = self.src_embedding(src) * self.scale
    x = self.positional_encoding(x)
    
    # Pass through encoder layers
    for layer in self.encoder_layers:
        x, _ = layer(x, src_mask)
    
    return self.encoder_norm(x)

def _decode(self, tgt, encoder_output, src_mask=None, tgt_mask=None):
    """Decode target sequence, attending to encoder output."""
    # Embed and add positional encoding
    x = self.tgt_embedding(tgt) * self.scale
    x = self.positional_encoding(x)
    
    # Pass through decoder layers
    for layer in self.decoder_layers:
        x, _ = layer(x, encoder_output, src_mask, tgt_mask)
    
    return self.decoder_norm(x)

# Attach methods to the class
Transformer.encode = _encode
Transformer.decode = _decode

print("✓ encode() and decode() methods added (Part 2/3)")
print("  encode: src tokens -> encoder representations")
print("  decode: tgt tokens + encoder output -> decoder representations")

In [ ]:
# === Building the Transformer: Part 3/3 — Putting It All Together ===
# The forward() method connects Encoder -> Decoder -> Output projection

def _forward(self, src, tgt, src_mask=None, tgt_mask=None):
    """Full forward pass: encode source, decode target, project to vocab."""
    # Encode source
    encoder_output = self.encode(src, src_mask)
    
    # Decode target
    decoder_output = self.decode(tgt, encoder_output, src_mask, tgt_mask)
    
    # Project to vocabulary
    logits = self.output_projection(decoder_output)
    
    return logits

# Attach forward method to the class
Transformer.forward = _forward

# Create a mini Transformer to test
model = Transformer(
    src_vocab_size=1000,
    tgt_vocab_size=1000,
    d_model=128,
    num_heads=4,
    num_encoder_layers=2,
    num_decoder_layers=2,
    d_ff=512,
    dropout=0.1
)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print("✓ Complete Transformer assembled! (3/3)")
print(f"  Total parameters: {total_params:,}")
print(f"  This is a ~{total_params/1e6:.1f}M parameter model")
print(f"  (GPT-3 has 175B parameters!)")

In [ ]:
# Test forward pass
batch_size = 4
src_len = 20
tgt_len = 15

# Random token indices
src = torch.randint(0, 1000, (batch_size, src_len))
tgt = torch.randint(0, 1000, (batch_size, tgt_len))

# Create causal mask for decoder
tgt_mask = create_causal_mask(tgt_len)

# Forward pass
model.train(False)
with torch.no_grad():
    logits = model(src, tgt, tgt_mask=tgt_mask)

print(f"Source shape: {src.shape}")
print(f"Target shape: {tgt.shape}")
print(f"Output logits shape: {logits.shape}")
print(f"\nEach position predicts distribution over {1000} tokens")

## 9. Training on a Simple Task: Copy Sequence

Let's train our Transformer on a simple task: copying a sequence. This verifies our implementation works!

In [ ]:
# Simple copy task: input [1, 5, 3, 2] -> output [1, 5, 3, 2]
# We use special tokens: 0=PAD, 1=BOS (begin), 2=EOS (end)

def generate_copy_data(batch_size, seq_len, vocab_size=50):
    """Generate data for the copy task."""
    # Random sequences (excluding special tokens 0, 1, 2)
    sequences = torch.randint(3, vocab_size, (batch_size, seq_len))
    
    # Source: just the sequence
    src = sequences
    
    # Target: BOS + sequence (for teacher forcing)
    bos = torch.ones(batch_size, 1, dtype=torch.long)  # BOS token = 1
    tgt_input = torch.cat([bos, sequences[:, :-1]], dim=1)
    
    # Target output: sequence (what we want to predict)
    tgt_output = sequences
    
    return src, tgt_input, tgt_output

# Generate a batch
src, tgt_input, tgt_output = generate_copy_data(batch_size=4, seq_len=8)
print("Source (to copy):")
print(src[0].numpy())
print("\nTarget input (teacher forcing):")
print(tgt_input[0].numpy())
print("\nTarget output (ground truth):")
print(tgt_output[0].numpy())

In [ ]:
# Train the copy task
vocab_size = 50
seq_len = 10
batch_size = 32
n_epochs = 100

# Create model
copy_model = Transformer(
    src_vocab_size=vocab_size,
    tgt_vocab_size=vocab_size,
    d_model=64,
    num_heads=4,
    num_encoder_layers=2,
    num_decoder_layers=2,
    d_ff=256,
    dropout=0.1
).to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(copy_model.parameters(), lr=0.001)

# Training loop
losses = []
accuracies = []

print("Training copy task...")
print("=" * 50)

for epoch in range(n_epochs):
    copy_model.train(True)
    
    # Generate batch
    src, tgt_input, tgt_output = generate_copy_data(batch_size, seq_len, vocab_size)
    src = src.to(device)
    tgt_input = tgt_input.to(device)
    tgt_output = tgt_output.to(device)
    
    # Create causal mask
    tgt_mask = create_causal_mask(seq_len).to(device)
    
    # Forward pass
    logits = copy_model(src, tgt_input, tgt_mask=tgt_mask)
    
    # Compute loss
    loss = criterion(logits.reshape(-1, vocab_size), tgt_output.reshape(-1))
    
    # Backward pass
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    
    # Track metrics
    losses.append(loss.item())
    predictions = logits.argmax(dim=-1)
    accuracy = (predictions == tgt_output).float().mean().item()
    accuracies.append(accuracy)
    
    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d}: Loss = {loss.item():.4f}, Accuracy = {accuracy:.2%}")

print(f"\nFinal: Loss = {losses[-1]:.4f}, Accuracy = {accuracies[-1]:.2%}")

In [ ]:
# Plot training progress
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(losses)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')

axes[1].plot(accuracies)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training Accuracy')
axes[1].set_ylim([0, 1.05])

plt.tight_layout()
plt.show()

In [ ]:
# Test the trained model
copy_model.train(False)

# Generate test sequence
test_src = torch.randint(3, vocab_size, (1, seq_len)).to(device)

# Autoregressive generation
def generate(model, src, max_len):
    """Generate sequence autoregressively."""
    model.train(False)
    
    # Encode source
    encoder_output = model.encode(src)
    
    # Start with BOS token
    generated = torch.ones(1, 1, dtype=torch.long, device=src.device)  # BOS = 1
    
    for _ in range(max_len):
        tgt_mask = create_causal_mask(generated.size(1)).to(src.device)
        
        decoder_output = model.decode(generated, encoder_output, tgt_mask=tgt_mask)
        logits = model.output_projection(decoder_output)
        
        # Get next token (greedy)
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        generated = torch.cat([generated, next_token], dim=1)
        
        # Stop at EOS
        if next_token.item() == 2:
            break
    
    return generated

with torch.no_grad():
    output = generate(copy_model, test_src, max_len=seq_len)

print("Copy Task Test:")
print("-" * 40)
print(f"Input:  {test_src[0].cpu().numpy()}")
print(f"Output: {output[0, 1:].cpu().numpy()}")  # Skip BOS
print(f"\nMatch: {torch.equal(test_src[0], output[0, 1:seq_len+1])}")

## 10. Visualizing Attention in Action

In [ ]:
# Get attention weights from encoder
copy_model.train(False)
test_src = torch.randint(3, vocab_size, (1, 8)).to(device)

# Hook to capture attention weights
attention_weights = []

def hook_fn(module, input, output):
    _, weights = output
    attention_weights.append(weights.detach().cpu())

# Register hooks
hooks = []
for layer in copy_model.encoder_layers:
    hooks.append(layer.attention.register_forward_hook(hook_fn))

# Forward pass
with torch.no_grad():
    _ = copy_model.encode(test_src)

# Remove hooks
for h in hooks:
    h.remove()

# Visualize attention from first layer
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
layer_weights = attention_weights[0][0]  # First layer, first sample

for head_idx in range(4):
    ax = axes[head_idx]
    ax.imshow(layer_weights[head_idx].numpy(), cmap='Blues')
    ax.set_title(f'Head {head_idx + 1}')
    ax.set_xlabel('Key Position')
    if head_idx == 0:
        ax.set_ylabel('Query Position')

plt.suptitle(f'Encoder Layer 1 Attention\nInput: {test_src[0].cpu().numpy()}', fontsize=12)
plt.tight_layout()
plt.show()

print("Different heads learn different attention patterns!")

## 11. Architecture Diagram

In [ ]:
# Visualize the transformer architecture
fig, ax = plt.subplots(1, 1, figsize=(14, 10))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')

# Helper function to draw boxes
def draw_box(ax, x, y, w, h, text, color='lightblue'):
    rect = plt.Rectangle((x, y), w, h, linewidth=2, edgecolor='black', facecolor=color)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center', fontsize=9, fontweight='bold')

def draw_arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.5))

# Encoder side
ax.text(3.5, 9.5, 'ENCODER', ha='center', fontsize=12, fontweight='bold')

draw_box(ax, 2, 0.5, 3, 0.6, 'Input Embedding', 'lightyellow')
draw_box(ax, 2, 1.3, 3, 0.6, '+ Positional Encoding', 'lightyellow')

# Encoder layer
draw_box(ax, 2, 2.2, 3, 0.8, 'Multi-Head\nSelf-Attention', 'lightblue')
draw_box(ax, 2, 3.2, 3, 0.4, 'Add & Norm', 'lightgreen')
draw_box(ax, 2, 3.8, 3, 0.8, 'Feed Forward', 'lightblue')
draw_box(ax, 2, 4.8, 3, 0.4, 'Add & Norm', 'lightgreen')

# Nx indicator
ax.text(5.3, 3.5, 'Nx', fontsize=14, fontweight='bold')
rect = plt.Rectangle((1.8, 2.1), 3.4, 3.2, linewidth=2, edgecolor='gray', 
                      facecolor='none', linestyle='--')
ax.add_patch(rect)

# Decoder side
ax.text(10.5, 9.5, 'DECODER', ha='center', fontsize=12, fontweight='bold')

draw_box(ax, 9, 0.5, 3, 0.6, 'Output Embedding', 'lightyellow')
draw_box(ax, 9, 1.3, 3, 0.6, '+ Positional Encoding', 'lightyellow')

# Decoder layer
draw_box(ax, 9, 2.2, 3, 0.8, 'Masked Multi-Head\nSelf-Attention', 'lightcoral')
draw_box(ax, 9, 3.2, 3, 0.4, 'Add & Norm', 'lightgreen')
draw_box(ax, 9, 3.8, 3, 0.8, 'Multi-Head\nCross-Attention', 'lightblue')
draw_box(ax, 9, 4.8, 3, 0.4, 'Add & Norm', 'lightgreen')
draw_box(ax, 9, 5.4, 3, 0.8, 'Feed Forward', 'lightblue')
draw_box(ax, 9, 6.4, 3, 0.4, 'Add & Norm', 'lightgreen')

# Nx indicator
ax.text(12.3, 4.5, 'Nx', fontsize=14, fontweight='bold')
rect = plt.Rectangle((8.8, 2.1), 3.4, 4.8, linewidth=2, edgecolor='gray', 
                      facecolor='none', linestyle='--')
ax.add_patch(rect)

# Output
draw_box(ax, 9, 7.2, 3, 0.6, 'Linear', 'lightblue')
draw_box(ax, 9, 8.0, 3, 0.6, 'Softmax', 'plum')
ax.text(10.5, 8.9, 'Output Probabilities', ha='center', fontsize=10)

# Cross-attention arrow
ax.annotate('', xy=(9, 4.2), xytext=(5.2, 4.2),
            arrowprops=dict(arrowstyle='->', color='blue', lw=2))
ax.text(7.1, 4.5, 'Encoder\nOutput', ha='center', fontsize=8, color='blue')

# Labels
ax.text(3.5, 0.1, 'Source Tokens', ha='center', fontsize=10)
ax.text(10.5, 0.1, 'Target Tokens\n(shifted right)', ha='center', fontsize=10)

plt.title('The Transformer Architecture', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

## 12. Exercises: Push Your Transformer Further

In [ ]:
# --- Exercise 1: Reverse Instead of Copy ---
# The Transformer learned to COPY sequences. Can it learn to REVERSE them?
# Input:  [1, 2, 3, 4, 5]
# Target: [5, 4, 3, 2, 1]

# YOUR CODE HERE:
# Generate reverse-task data (modify the data generation to reverse targets)
def generate_reverse_data(num_samples=500, seq_len=10, vocab_size=50):
    """Generate input-output pairs where output is input reversed."""
    src = torch.randint(3, vocab_size, (num_samples, seq_len))  # skip 0,1,2 for special tokens
    tgt = src.flip(dims=[1])  # Reverse each sequence
    return src, tgt

rev_src, rev_tgt = generate_reverse_data()

# Verify the data looks right
print("Sample input: ", rev_src[0].tolist())
print("Sample target:", rev_tgt[0].tolist())
assert rev_tgt[0].tolist() == rev_src[0].tolist()[::-1], "Target should be reversed input!"
print("\n✓ Data generated correctly!")

# --- Now train on it! ---
# Create a new Transformer (same architecture as the copy task)
reverse_model = Transformer(
    src_vocab_size=50, tgt_vocab_size=50, d_model=64,
    num_heads=4, num_encoder_layers=2, num_decoder_layers=2,
    d_ff=256, dropout=0.1
).to(device)
reverse_optimizer = torch.optim.Adam(reverse_model.parameters(), lr=0.001)
reverse_criterion = nn.CrossEntropyLoss()

# For training, we need teacher-forcing targets: BOS + reversed[:-1]
bos = torch.ones(100, 1, dtype=torch.long, device=device)
rev_src_train = rev_src[:100].to(device)
rev_tgt_out = rev_tgt[:100].to(device)
rev_tgt_in = torch.cat([bos, rev_tgt_out[:, :-1]], dim=1)

print("\nTraining Transformer to reverse sequences...")
for epoch in range(30):
    reverse_model.train(True)
    reverse_optimizer.zero_grad()
    tgt_mask = create_causal_mask(rev_tgt_in.size(1)).to(device)
    output = reverse_model(rev_src_train, rev_tgt_in, tgt_mask=tgt_mask)
    loss = reverse_criterion(
        output.reshape(-1, output.size(-1)),
        rev_tgt_out.reshape(-1)
    )
    loss.backward()
    reverse_optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1}: loss = {loss.item():.4f}")

print("\n💡 Reversing is HARDER than copying!")
print("   The model must learn long-range position mapping (first → last)")
print("   If loss is still high after 30 epochs, it needs more training.")
print("\nExercise 1 passed! ✓")

In [ ]:
# --- Exercise 2: Count Parameters ---
# How many trainable parameters does your Transformer have?
total_params = sum(p.numel() for p in reverse_model.parameters() if p.requires_grad)
print(f"Your Transformer has {total_params:,} trainable parameters")

# For reference: GPT-3 has 175,000,000,000 (175 billion)
ratio = 175_000_000_000 / total_params
print(f"GPT-3 is {ratio:,.0f}x larger than your model")

print("\nExercise 2 passed! ✓")
print("\n🎉 All exercises passed!")

## Summary

We built a Transformer from scratch with these key components:

**Core Building Blocks:**
- **Positional Encoding**: Sinusoidal functions inject position info
- **Scaled Dot-Product Attention**: Q, K, V mechanism with scaling
- **Multi-Head Attention**: Multiple parallel attention mechanisms
- **Feed-Forward Network**: Position-wise MLP
- **Layer Normalization**: Stabilizes training
- **Residual Connections**: Enable gradient flow

**Architecture:**
- **Encoder**: Self-attention + FFN blocks
- **Decoder**: Masked self-attention + Cross-attention + FFN blocks

**Key Insights:**
- Attention lets each token "see" all other tokens
- Multi-head attention captures different relationship types
- Causal masking enables autoregressive generation
- Cross-attention connects encoder and decoder

**Next up**: Using pre-trained Transformers with Hugging Face!